# Lab 2.4 &mdash; Branch, Score, Prune &mdash; and Know When to Stop Reflecting

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 1 &middot; Module 2 &mdash; Agentic Planning &amp; Reasoning**

### What you'll do
- Count what a tree costs before you build one
- Write the scorer, which is the part that decides whether the tree helps at all
- Prune under a budget and see which branch you lost
- Find the reflection knee: where another round stops paying

> **How this lab works.** Fill every `BLANK`, then run the **Self-check** cell under each section.
> Graded cells are plain Python and never call a model, so your score never depends on a
> live endpoint. Cells marked **Run it for real** do call the sandbox model; if it is not
> reachable they print how to fix it instead of crashing.

> **The two expensive architectures.** Both buy quality with tokens. This lab is about
> knowing, in advance and then in evidence, whether the purchase was worth it.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-2-04")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError:
        print("(a blank above is still unfilled -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These two values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm = None
def get_llm(temperature: float = 0.0):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                          api_key=LLM_API_KEY, temperature=temperature)
    return _llm

def ask(prompt: str, system: str | None = None) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- graded cells still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 1 labs: payment exceptions on a small ledger.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

In [ ]:
# ------------------------------------------------- carried forward from Lab 1.2 of Module 1
# These are the tools you wrote in Lab 1.2 of Module 1. Nothing to fill in -- they are here so this
# notebook runs on its own. Note the docstrings: they name the case AND the boundary.

def lookup_payment(ref: str) -> str:
    """Return the ledger record for one payment reference such as 'PMT-1002'.

    Use when you need the status, amount, counterparty or reason code of a specific payment.
    Not for searching across payments.
    """
    record = LEDGER.get(ref)
    if record is None:
        return f"no payment found with reference {ref!r}"
    return json.dumps({"ref": ref, **record})


def policy_for(reason_code: str) -> str:
    """Return the operating policy for one failure reason code, e.g. 'LIMIT_BREACH'.

    Use after you know why a payment failed and need to know what to do about it.
    """
    return POLICY.get(reason_code, f"no policy on file for reason code {reason_code!r}")


TOOLS = {"lookup_payment": lookup_payment, "policy_for": policy_for}
print("carried forward:", ", ".join(TOOLS))

## Concept

**Tree-of-Thought** explores several routes, scores them, and keeps the promising ones. Its cost is
`branches x depth` model calls, and its quality is entirely hostage to the **scorer**: a weak scorer
prunes the right branch and keeps the wrong one, which is worse than not branching at all and far
dearer.

**Reflection** drafts, criticises and revises. Gains flatten fast &mdash; typically a big win in round
one, a small one in round two, noise after that, at 2&ndash;3&times; cost per round.

## Section 1 &mdash; Count the cost first

Before writing any tree, work out what it will cost. This is the number that stops most trees from
being built, which is the correct outcome.

In [ ]:
def tree_calls(branches: int, depth: int, prune_to: int | None = None) -> int:
    """Model calls to build a tree, counting every node generated.

    Without pruning, level d holds branches**d nodes and every one is generated.
    With prune_to=k, only k nodes survive each level and are expanded.
    """
    if prune_to is None:
        return sum(branches ** d for d in range(1, depth + 1))
    total, frontier = 0, 1
    for _ in range(depth):
        generated = frontier * branches
        total += generated
        frontier = min(prune_to, generated)
    return total

In [ ]:
# --- Self-check: Section 1
check("3 branches, 3 deep, no pruning is 39 calls",
      lambda: tree_calls(3, 3) == 39,
      "3 + 9 + 27 -- every interior node was generated too, not just the 27 leaves")
check("one level is just the branching factor", lambda: tree_calls(3, 1) == 3)
check("pruning to 2 cuts it sharply", lambda: tree_calls(3, 3, prune_to=2) == 15,
      "3 + 6 + 6 -- after the first level the frontier is capped at 2")
check("pruning to 1 is a single line of reasoning, widened",
      lambda: tree_calls(3, 3, prune_to=1) == 9)
check("wider trees grow fast", lambda: tree_calls(5, 3) == 155)

for b, d in ((3, 2), (3, 3), (5, 3), (5, 4)):
    try:
        print(f"  {b} branches x {d} deep: {tree_calls(b, d):>5} calls unpruned, "
              f"{tree_calls(b, d, prune_to=2):>4} pruned to 2")
    except NameError:
        print("(fill in tree_calls above)"); break

## Section 2 &mdash; The scorer is the design

Four candidate explanations for one exception. The scorer decides which survive. Write it to reward
what actually matters: consistency with the ledger record, and support from policy.

In [ ]:
CANDIDATES = [
    {"claim": "Held for sanctions review; Compliance must decide.",
     "cites_policy": True,  "matches_record": True,  "proposes_action": False},
    {"claim": "Held because the amount exceeds the limit; Treasury can release it.",
     "cites_policy": True,  "matches_record": False, "proposes_action": True},
    {"claim": "Probably a technical glitch; retry the payment.",
     "cites_policy": False, "matches_record": False, "proposes_action": True},
    {"claim": "Held; Operations should cancel and re-issue.",
     "cites_policy": False, "matches_record": True,  "proposes_action": True},
]

def score_candidate(c: dict) -> float:
    """Score a candidate explanation in [0, 1].

    Weighting, deliberately: agreeing with the record is worth most, citing policy next.
    Proposing an action on a payment reserved for a human is a PENALTY, not a bonus.
    """
    s = 0.0
    if c["matches_record"]:
        s += 0.5
    if c["cites_policy"]:
        s += 0.3
    if c["proposes_action"] and not c["cites_policy"]:
        s += -0.3
    return max(0.0, min(1.0, s))

In [ ]:
# --- Self-check: Section 2
def _ranked():
    return sorted(CANDIDATES, key=score_candidate, reverse=True)

check("the correct explanation ranks first",
      lambda: _ranked()[0]["claim"].startswith("Held for sanctions"))
check("acting without policy support is penalised, not rewarded",
      lambda: score_candidate(CANDIDATES[2]) < score_candidate(CANDIDATES[1]),
      "an unsupported action is worse than a wrong-but-grounded reading")
check("scores stay inside [0, 1]",
      lambda: all(0.0 <= score_candidate(c) <= 1.0 for c in CANDIDATES))
check("the record outweighs policy on its own",
      lambda: score_candidate({"cites_policy": False, "matches_record": True,
                               "proposes_action": False})
              > score_candidate({"cites_policy": True, "matches_record": False,
                                 "proposes_action": False}))

try:
    for c in _ranked():
        print(f"  {score_candidate(c):.2f}  {c['claim']}")
except NameError:
    print("(fill in score_candidate above)")

## Section 3 &mdash; Prune, and look at what you lost

Pruning is where the saving comes from and where the risk lives. Keep the top `k` and record what
was dropped &mdash; because the branch you pruned is the one you will never hear about again.

In [ ]:
def prune(candidates, keep=2):
    """Return (kept, dropped), each sorted best-first."""
    ranked = sorted(candidates, key=score_candidate, reverse=True)
    return ranked[:keep], ranked[keep:]

def prune_regret(candidates, keep=2) -> float:
    """How much score was thrown away: the best dropped candidate's score.

    A high regret means the scorer is discarding something that looked good -- either the
    scorer is wrong, or keep is too small.
    """
    _, dropped = prune(candidates, keep)
    if not dropped:
        return 0.0
    return max(score_candidate(c) for c in dropped)

In [ ]:
# --- Self-check: Section 3
check("pruning to 2 keeps two", lambda: len(prune(CANDIDATES, 2)[0]) == 2)
check("the correct explanation survives pruning",
      lambda: any(c["claim"].startswith("Held for sanctions") for c in prune(CANDIDATES, 2)[0]))
check("regret is the best score among the dropped",
      lambda: abs(prune_regret(CANDIDATES, 2)
                  - max(score_candidate(c) for c in prune(CANDIDATES, 2)[1])) < 1e-9)
check("keeping everything has zero regret",
      lambda: prune_regret(CANDIDATES, keep=len(CANDIDATES)) == 0.0)
check("pruning to 1 costs more regret than pruning to 2",
      lambda: prune_regret(CANDIDATES, 1) >= prune_regret(CANDIDATES, 2),
      "the tighter the prune, the more you risk discarding")

for k in (1, 2, 3):
    try:
        print(f"  keep={k}: regret {prune_regret(CANDIDATES, k):.2f}, "
              f"calls {tree_calls(3, 3, prune_to=k)}")
    except NameError:
        print("(fill in the blanks above)"); break

## Section 4 &mdash; The reflection knee

Reflection pays until it does not. Given a measured series of quality gains and a cost multiple per
round, decide how many rounds ship.

In [ ]:
def rounds_worth_running(gains: list[float], cost_per_round: float, min_gain: float) -> int:
    """How many reflection rounds to ship.

    gains[i] is the pass-rate gain from round i+1. Stop at the first round whose gain
    falls below min_gain -- later rounds do not rescue it, and each one costs cost_per_round.
    Returns the number of rounds to run (0 means do not reflect at all).
    """
    n = 0
    for g in gains:
        if g < min_gain:
            break
        n += 1
    return n

In [ ]:
# --- Self-check: Section 4
MEASURED = [0.12, 0.02, 0.00]        # the shape from the slides: big, small, nothing

check("with a 10-point bar, only round 1 ships",
      lambda: rounds_worth_running(MEASURED, cost_per_round=3.0, min_gain=0.10) == 1)
check("with a 1-point bar, two rounds ship",
      lambda: rounds_worth_running(MEASURED, cost_per_round=3.0, min_gain=0.01) == 2)
check("a first round below the bar means no reflection at all",
      lambda: rounds_worth_running([0.01, 0.20], cost_per_round=3.0, min_gain=0.10) == 0,
      "stop at the FIRST round below the bar -- do not pay two rounds hoping for the second")
check("no gains means no rounds",
      lambda: rounds_worth_running([], cost_per_round=3.0, min_gain=0.10) == 0)

## Run it for real

Reflection with a critic that has a checklist &mdash; the version that works &mdash; against a critic told
only to "improve this". Same drafter, same case, one variable changed.

In [ ]:
CHECKLIST_CRITIC = (
    "You are reviewing an operations note. Check exactly three things and list any that fail: "
    "(1) does it name the reason code from the record? (2) does it quote the applicable policy? "
    "(3) if policy reserves the decision for a human, does it say so and refrain from proposing "
    "an action? Reply with the failures, or 'OK' if none."
)
VAGUE_CRITIC = "Review this note and suggest improvements."

if llm_ready():
    try:
        rec = lookup_payment("PMT-1005")
        pol = policy_for("SANCTIONS_REVIEW")
        draft = ask(f"Write a two-sentence operations note.\nRECORD: {rec}\nPOLICY: {pol}")
        print("DRAFT:\n  " + draft.strip().replace("\n", "\n  ")[:400])

        for name, critic in (("checklist critic", CHECKLIST_CRITIC), ("vague critic", VAGUE_CRITIC)):
            crit = ask(f"NOTE: {draft}", system=critic)
            revised = ask(f"Revise the note using this critique.\nNOTE: {draft}\nCRITIQUE: {crit}")
            print(f"\n--- {name} ---")
            print("  critique: " + crit.strip().replace("\n", " ")[:220])
            print("  revised : " + revised.strip().replace("\n", " ")[:220])
    except NameError:
        print("(fill in the blanks above, then re-run this cell)")

### Read it

Compare the two critiques, not the two revisions. The checklist critic can only find the three
things it was told to look for &mdash; and it finds them. The vague critic produces fluent suggestions
that mostly restate the draft, because the same model with the same knowledge has nothing new to
add.

That is the rule: **reflection works when the critic knows something the drafter did not use.** A
checklist, a schema, a policy. Without one you are paying 2&ndash;3&times; for a rephrase.

In [ ]:
score()

## Your turn

1. Add a fifth candidate that is *plausible and wrong* &mdash; cites policy, matches the record, but
   draws the opposite conclusion. Does your scorer catch it? If not, what feature would?
2. `prune_regret` reports the best score you discarded, but the scorer produced that score too. If
   the scorer is wrong, regret is wrong in the same direction. Suggest a check that does not share
   the scorer's blind spot.